# Limpeza de `feed_posts.csv`

**Objetivo:** gerar os posts da rede social prontos para análise em `dados/processados/feed_posts_limpa.parquet`.

**Regras aplicadas** (decisões de 07/11/2026; o diagnóstico está no notebook 19):

| Passo | Regra |
|---|---|
| 1 | `data_hora` como data e hora em Brasília; `data` |
| 2 | Tirar os posts depois de 30/09/2026 |
| 3 | `[post removido]` vira texto ausente, com a marca `post_removido` (o post continua contando) |
| 4 | `curtidas` como número inteiro |

**Atenção ao usar:** as curtidas são as acumuladas até a exportação, sem data, e podem incluir curtidas dadas depois do
corte: não use como preditor de saída; prefira o número de posts. Posts removidos não vão para a extração.

Nada em `dados/brutos/` é alterado: o hash dos arquivos é conferido no início e no fim.

In [1]:
ARQUIVO = "feed_posts.csv"
NOME_SAIDA = "feed_posts_limpa.parquet"

import hashlib
import re
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 70)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BRUTOS = ROOT / "dados" / "brutos"
PROCESSADOS = ROOT / "dados" / "processados"
SAIDAS = ROOT / "saidas"
CORTE = pd.Timestamp("2026-09-30")
FIM_CORTE = CORTE + pd.Timedelta(days=1)

ARQUIVOS = [ARQUIVO]

def sha1(path):
    return hashlib.sha1(path.read_bytes()).hexdigest()

hash_inicio = {f: sha1(BRUTOS / f) for f in ARQUIVOS}

registro = []

def registrar(passo, regra, df, alteradas=0):
    anterior = registro[-1]["linhas"] if registro else len(df)
    registro.append({"passo": passo, "regra": regra, "linhas": len(df), "removidas": anterior - len(df),
                     "alteradas": int(alteradas)})
    print(f"{passo}: {len(df):,} linhas ({anterior - len(df):,} removidas, {int(alteradas):,} alteradas)")
    return df

SEM_CONTEUDO = {"nada", "ok", "x", ".", "-", "asdf", "kkkk", "sla", "nao sei", "não sei", "nan", "null", "none", "n/a"}

def sem_conteudo(s):
    t = s.str.strip().str.lower()
    so_simbolo = t.ne("") & ~t.str.contains(r"[a-z0-9à-ú]", regex=True)
    return t.isin(SEM_CONTEUDO) | so_simbolo

usuarios = pd.read_parquet(PROCESSADOS / "usuarios_limpa.parquet", columns=["user_id", "data_cadastro", "canal_aquisicao"]).set_index("user_id")

## 0. Leitura

In [2]:
feed = pd.read_csv(BRUTOS / ARQUIVO, dtype=str, keep_default_na=False)
assert len(feed) == 119_309 and feed["post_id"].is_unique
base = registrar("0. Arquivo bruto", "feed_posts.csv sem filtro", feed.copy())

0. Arquivo bruto: 119,309 linhas (0 removidas, 0 alteradas)


## 1. Data e hora em Brasília

In [3]:
base["data_hora"] = pd.to_datetime(base["data_hora"], format="%Y-%m-%d %H:%M:%S")
base["data"] = base["data_hora"].dt.normalize()
assert base["data_hora"].notna().all()
base = registrar("1. Data e hora", "data_hora como horário de Brasília; data", base, len(base))

1. Data e hora: 119,309 linhas (0 removidas, 119,309 alteradas)


## 2. Posts depois do corte

In [4]:
depois = base["data_hora"] >= FIM_CORTE
print("depois do corte, por dia:", base.loc[depois, "data"].dt.strftime("%d/%m").value_counts().sort_index().to_dict())
assert depois.sum() == 122
base = registrar("2. Sem posts depois do corte", "data_hora até 30/09/2026", base.loc[~depois].copy())

depois do corte, por dia: {'01/10': 62, '02/10': 31, '03/10': 16, '04/10': 5, '05/10': 2, '06/10': 4, '07/10': 2}
2. Sem posts depois do corte: 119,187 linhas (122 removidas, 0 alteradas)


## 3. Posts removidos

In [5]:
removido = base["texto"].str.strip() == "[post removido]"
base["post_removido"] = removido
base["texto"] = base["texto"].where(~removido, None)
print(f"posts removidos (até o corte): {removido.sum():,}")
assert removido.sum() == 1_213
base = registrar("3. Posts removidos", "[post removido] vira texto ausente; marca post_removido", base, removido.sum())

posts removidos (até o corte): 1,213
3. Posts removidos: 119,187 linhas (0 removidas, 1,213 alteradas)


## 4. Curtidas

In [6]:
base["curtidas"] = pd.to_numeric(base["curtidas"]).astype("int64")
assert (base["curtidas"] >= 0).all()
base = registrar("4. Curtidas", "curtidas como inteiro", base, len(base))

4. Curtidas: 119,187 linhas (0 removidas, 119,187 alteradas)


## 5. Conferências

In [7]:
conferencias = {
    "linhas": (len(base), 119_187),
    "post_id repetido": (int(base["post_id"].duplicated().sum()), 0),
    "autores fora de usuarios_limpa": (int((~base["user_id"].isin(usuarios.index)).sum()), 0),
    "post antes do dia do cadastro": (int((base["data"] < base["user_id"].map(usuarios["data_cadastro"])).sum()), 0),
    "post depois do corte": (int((base["data"] > CORTE).sum()), 0),
    "texto ausente fora dos removidos": (int((base["texto"].isna() & ~base["post_removido"]).sum()), 0),
}

In [8]:
tabela = pd.DataFrame(conferencias, index=["obtido", "esperado"]).T
tabela["ok"] = tabela["obtido"] == tabela["esperado"]
display(tabela)
assert tabela["ok"].all()

,obtido,esperado,ok
linhas,119187,119187,True
post_id repetido,0,0,True
autores fora de usuarios_limpa,0,0,True
post antes do dia do cadastro,0,0,True
post depois do corte,0,0,True
texto ausente fora dos removidos,0,0,True


## 6. Saídas e integridade dos brutos

In [9]:
COLUNAS = ["post_id", "user_id", "data_hora", "data", "texto", "post_removido", "curtidas"]
limpa = base.sort_values(["user_id", "data_hora", "post_id"])[COLUNAS].reset_index(drop=True)

for col in limpa.columns[limpa.dtypes == object]:
    limpa[col] = limpa[col].astype(object).where(limpa[col].notna(), None)   # ausente sempre como None

PROCESSADOS.mkdir(parents=True, exist_ok=True)
destino = PROCESSADOS / NOME_SAIDA
limpa.to_parquet(destino, index=False)
relida = pd.read_parquet(destino)
pd.testing.assert_frame_equal(relida, limpa)
pd.DataFrame(registro).to_csv(SAIDAS / f"limpeza_{Path(ARQUIVO).stem}_registro.csv", index=False)

print(f"{destino.relative_to(ROOT)}: {len(limpa):,} linhas x {limpa.shape[1]} colunas")
display(limpa.dtypes.rename("tipo").to_frame().assign(ausentes=limpa.isna().sum()))
display(pd.DataFrame(registro))

hash_fim = {f: sha1(BRUTOS / f) for f in ARQUIVOS}
assert hash_fim == hash_inicio
print("dados/brutos/ inalterado:", hash_fim == hash_inicio)

dados\processados\feed_posts_limpa.parquet: 119,187 linhas x 7 colunas


,tipo,ausentes
post_id,object,0
user_id,object,0
data_hora,datetime64[ns],0
data,datetime64[ns],0
texto,object,1213
post_removido,bool,0
curtidas,int64,0


,passo,regra,linhas,removidas,alteradas
0,0. Arquivo bruto,feed_posts.csv sem filtro,119309,0,0
1,1. Data e hora,data_hora como horário de Brasília; data,119309,0,119309
2,2. Sem posts depois do corte,data_hora até 30/09/2026,119187,122,0
3,3. Posts removidos,[post removido] vira texto ausente; marca post_removido,119187,0,1213
4,4. Curtidas,curtidas como inteiro,119187,0,119187


dados/brutos/ inalterado: True
